# XGBoost
Accompanies the notes on **Extreme Gradient Boosting (XGBoost)**.

Needs the `xgboost` package (`pip install xgboost`).

## 1. The worked example, checked with the library

In [ ]:
import numpy as np
import xgboost as xgb

x = np.array([[1.5], [2.5], [4.0], [6.0]]); y = np.array([0, 0, 1, 1])
for mcw in [1, 0]:
    model = xgb.XGBClassifier(n_estimators=1, max_depth=1, learning_rate=0.3, reg_lambda=1.0, gamma=0.25,
                              min_child_weight=mcw, base_score=0.5)
    model.fit(x, y)
    margin = model.predict(x, output_margin=True)          # F_1 = F_0 + eta * w, on the log-odds scale
    print(f"min_child_weight={mcw}: F1 = {margin.round(4)}, p = {model.predict_proba(x)[:, 1].round(4)}")
print(model.get_booster().trees_to_dataframe()[["Node", "Feature", "Split", "Gain", "Cover"]])

## 2. Bank data: train / validation / test

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
# XGBoost does not accept "[", "]" or "<" in column names (some dummies have them)
X.columns = X.columns.str.replace(r"[\[\]<]", "_", regex=True)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
x_fit, x_val, y_fit, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

## 3. A first model with early stopping on validation data

In [ ]:
model = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8,
                          colsample_bytree=0.8, early_stopping_rounds=50, eval_metric="logloss", random_state=0)
model.fit(x_fit, y_fit, eval_set=[(x_val, y_val)], verbose=False)     # validation data, not the test set
print("best number of trees:", model.best_iteration + 1)
print("validation log-loss:", round(model.best_score, 4))
print("test AUC:", round(roc_auc_score(y_test, model.predict_proba(x_test)[:, 1]), 4))

## 4. How the regularisation parameters behave

In [ ]:
def val_loss(**params):
    m = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          early_stopping_rounds=50, eval_metric="logloss", random_state=0, **params)
    m.fit(x_fit, y_fit, eval_set=[(x_val, y_val)], verbose=False)
    return round(m.best_score, 4), m.best_iteration + 1

for depth in [2, 4, 6, 10]:
    print(f"max_depth={depth:2d}            : (validation log-loss, trees) = {val_loss(max_depth=depth)}")
for lam in [0, 1, 10, 100]:
    print(f"max_depth=10, reg_lambda={lam:3d}: {val_loss(max_depth=10, reg_lambda=lam)}")
for g in [0, 1, 5]:
    print(f"max_depth=10, gamma={g}         : {val_loss(max_depth=10, gamma=g)}")
for mcw in [1, 5, 20]:
    print(f"max_depth=10, min_child_weight={mcw:2d}: {val_loss(max_depth=10, min_child_weight=mcw)}")

## 5. scale_pos_weight

In [ ]:
ratio = (y_fit == 0).sum() / (y_fit == 1).sum()
print("negatives / positives:", round(ratio, 2))
for spw in [1, ratio]:
    m = xgb.XGBClassifier(n_estimators=model.best_iteration + 1, learning_rate=0.05, max_depth=4, subsample=0.8,
                          colsample_bytree=0.8, scale_pos_weight=spw, random_state=0).fit(x_train, y_train)
    p = m.predict_proba(x_test)[:, 1]
    print(f"scale_pos_weight={spw:5.2f}: test AUC {roc_auc_score(y_test, p):.4f}, mean predicted probability {p.mean():.3f}"
          f" (actual rate {y_test.mean():.3f}), log-loss {log_loss(y_test, p):.4f}")

## 6. Missing values without imputation (loan data)

In [ ]:
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld.dropna(subset=["Interest.Rate"])
print("missing values per column:", ld[cols].isna().sum().to_dict())
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
xr_fit, xr_val, yr_fit, yr_val = train_test_split(xr_train, yr_train, test_size=0.2, random_state=0)
reg = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=3, subsample=0.8,
                       early_stopping_rounds=50, random_state=0)
reg.fit(xr_fit, yr_fit, eval_set=[(xr_val, yr_val)], verbose=False)    # NaNs go in as they are
print("trees:", reg.best_iteration + 1, " test MAE:", round(mean_absolute_error(yr_test, reg.predict(xr_test)), 3))

## 7. Feature importance: gain vs weight

In [ ]:
imp = pd.DataFrame({"gain": model.get_booster().get_score(importance_type="gain"),
                    "weight (number of splits)": model.get_booster().get_score(importance_type="weight")})
print(imp.sort_values("gain", ascending=False).round(1).head(8))

## Try it yourself
1. Section 1 with `reg_lambda=0, gamma=0`.
2. `val_loss(max_depth=6, reg_lambda=10)` in section 4.